# Dealer → Brand-Model Ranking

Goal:
For each dealer, rank brand-model items by probability of future view within horizon H,
using only history strictly before cutoff t.


# Imports



In [70]:
import numpy as np
import pandas as pd
import pyarrow.csv as pacsv
import ast
import pyarrow as pa
from datetime import timedelta
from pathlib import Path

# Config

In [47]:
CATALOG_PATH = "brand_model_english_persian_ids.csv"
VIEW_FOLDER = Path("l2r_dataset/views")
BID_FOLDER = Path("l2r_dataset/bids")

HORIZON_DAYS = 7
TOP_N_POPULAR = 100

K_LIST = [5, 10]

BUSINESS_KEY = ["dealer_id", "auction_id", "brand_id", "model_id", "created_time"]
ITEM_KEY_COLS = ["brand_id", "model_id"]


# Load data

In [48]:
def read_folder_csvs(folder):
    tables = [pacsv.read_csv(file) for file in sorted(folder.glob("*.csv"))]
    return pa.concat_tables(tables).to_pandas()
catalog = pd.read_csv(CATALOG_PATH)
views = read_folder_csvs(VIEW_FOLDER)
bids = read_folder_csvs(BID_FOLDER)

print("catalog:", catalog.shape)
print("bids:", bids.shape)
print("views:", views.shape)

catalog: (669, 7)
bids: (1139590, 6)
views: (12139008, 6)


# Standardize schema

In [49]:
def standardize_events(df, event_name):
    df = df.copy()
    df["created_time"] = pd.to_datetime(df["created_time"], utc=True).dt.floor("min")
    df["item_key"] = list(zip(df["brand_id"], df["model_id"]))
    return df

bids = standardize_events(bids, "bid")
views = standardize_events(views, "view")


# Dedup policy

In [52]:
def dedup_with_count(df, count_col):
    df = df.copy()

    df[count_col] = (
        df.groupby(BUSINESS_KEY)["dealer_id"]
          .transform("size")
    )

    df = df.drop_duplicates(subset=BUSINESS_KEY, keep="first")

    return df

In [53]:
bids_dedup = dedup_with_count(bids, "bid_in_minute")
views_dedup = dedup_with_count(views, "view_in_minute")

print("bids raw   :", bids.shape, "-> dedup:", bids_dedup.shape)
print("views raw  :", views.shape, "-> dedup:", views_dedup.shape)

bids raw   : (1139590, 7) -> dedup: (994950, 8)
views raw  : (12139008, 7) -> dedup: (3129885, 8)


# Sanity checks

In [54]:
def quick_profile(name, df):
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("time range:", df["created_time"].min(), "->", df["created_time"].max())
    print("dealers:", df["dealer_id"].nunique())
    print("auctions:", df["auction_id"].nunique())
    print("items:", df[ITEM_KEY_COLS].drop_duplicates().shape[0])
    print("nulls:\n", df.isna().sum())
    print()

quick_profile("bids_dedup", bids_dedup)
quick_profile("views_dedup", views_dedup)


--- bids_dedup ---
shape: (994950, 8)
time range: 2026-07-01 00:01:00+00:00 -> 2026-08-17 15:30:00+00:00
dealers: 6789
auctions: 37322
items: 366
nulls:
 bid_id           0
dealer_id        0
auction_id       0
brand_id         0
model_id         0
created_time     0
item_key         0
bid_in_minute    0
dtype: int64

--- views_dedup ---
shape: (3129885, 8)
time range: 2026-07-01 00:00:00+00:00 -> 2026-08-17 15:25:00+00:00
dealers: 11803
auctions: 33391
items: 366
nulls:
 view_id           0
dealer_id         0
auction_id        0
brand_id          0
model_id          0
created_time      0
item_key          0
view_in_minute    0
dtype: int64



# Helpers for time split

In [55]:
def split_by_cutoff(df, cutoff, horizon_days):
    history = df[df["created_time"] < cutoff].copy()
    future_end = cutoff + pd.Timedelta(days=horizon_days)
    future = df[(df["created_time"] >= cutoff) & (df["created_time"] < future_end)].copy()
    return history, future

def build_future_labels(future_views):
    return (
        future_views[["dealer_id", "brand_id", "model_id"]]
        .drop_duplicates()
        .assign(label=1)
    )

def add_item_key(df):
    df = df.copy()
    df["item_key"] = list(zip(df["brand_id"], df["model_id"]))
    return df


# Helpers for  E2

In [71]:
def build_e2_candidates(history_views):
    touched = (
        history_views.groupby("dealer_id")["item_key"]
        .apply(set)
        .to_dict()
    )

    top_global = (
        history_views.groupby('item_key')
        .size()
        .sort_values(ascending=False)
        .head(TOP_N_POPULAR)
        .index
        .tolist()
    )
    
    candidates = []
    for d_id, items in touched.items():
        top_k_global = top_global

        # Top k based on touched item per dealer
        if len(items) < 10:
            top_k_global = top_global[:TOP_N_POPULAR//10]
        elif len(items) < 20:
            top_k_global = top_global[:TOP_N_POPULAR//5]
        elif len(items) < 30:
            top_k_global = top_global[:TOP_N_POPULAR//2]
        
        combined = items.union(set(top_k_global)) 
        for i_id in combined:
            candidates.append({"dealer_id": d_id, "item_id": i_id})

    df = pd.DataFrame(candidates)
    df['item_id'] = df['item_id'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    df[['brand_id', 'model_id']] = pd.DataFrame(df['item_id'].tolist(),index=df.index)
    df = df.drop(columns='item_id')
    return df


# Implementing baselines

# Baseline A1

In [77]:
def score_a1_dealer_history(history_views, history_bids, cutoff):
    view_stats = (
        history_views.groupby(["dealer_id"] + ITEM_KEY_COLS)
        .agg(
            past_view_count=("item_key", "count"),
            last_view_time=("created_time", "max"),
        )
        .reset_index()
    )

    bid_stats = (
        history_bids.groupby(["dealer_id"] + ITEM_KEY_COLS)
        .agg(
            past_bid_count=("item_key", "count"),
            last_bid_time=("created_time", "max"),
        )
        .reset_index()
    )

    scores = view_stats.merge(
        bid_stats,
        on=["dealer_id"] + ITEM_KEY_COLS,
        how="outer"
    )

    for col in ["past_view_count", "past_bid_count"]:
        scores[col] = scores[col].fillna(0)

    scores["view_recency_days"] = (cutoff - scores["last_view_time"]).dt.total_seconds() / 86400
    scores["bid_recency_days"] = (cutoff - scores["last_bid_time"]).dt.total_seconds() / 86400

    scores["view_recency_days"] = scores["view_recency_days"].fillna(999)
    scores["bid_recency_days"] = scores["bid_recency_days"].fillna(999)

    # simple, explainable heuristic
    scores["score_a1"] = (
        1.0 * scores["past_view_count"]
        + 10.5 * scores["past_bid_count"]
        - 0.05 * scores["view_recency_days"]
        - 0.05 * scores["bid_recency_days"]
    )

    return scores


# Baseline A2

In [78]:
def score_a2_popularity(history_views):
    scores = (
        history_views.groupby(ITEM_KEY_COLS)
        .size()
        .reset_index(name="score_a2")
    )
    return scores


# Baseline A3

In [79]:
def normalize_series(s):
    s = s.astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def score_a3_blend(scores_df, alpha=0.7):
    df = scores_df.copy()
    df["score_a1_norm"] = normalize_series(df["score_a1"].fillna(df["score_a1"].min() if "score_a1" in df else 0))
    df["score_a2_norm"] = normalize_series(df["score_a2"].fillna(0))
    df["score_a3"] = alpha * df["score_a1_norm"] + (1 - alpha) * df["score_a2_norm"]
    return df


# Metrics for evaluation

In [80]:
def recall_at_k(pred_items, true_items, k):
    pred_topk = pred_items[:k]
    true_set = set(true_items)
    if len(true_set) == 0:
        return np.nan
    return len(set(pred_topk) & true_set) / len(true_set)

def ndcg_at_k(pred_items, true_items, k):
    true_set = set(true_items)
    if len(true_set) == 0:
        return np.nan

    dcg = 0.0
    for i, item in enumerate(pred_items[:k], start=1):
        rel = 1 if item in true_set else 0
        dcg += rel / np.log2(i + 1)

    ideal_hits = min(len(true_set), k)
    idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else np.nan

def build_ground_truth(future_views):
    gt = (
        future_views[["dealer_id"] + ITEM_KEY_COLS]
        .drop_duplicates()
        .assign(item_key=lambda x: list(zip(x["brand_id"], x["model_id"])))
        .groupby("dealer_id")
        .agg(true_items=("item_key", list))
        .reset_index()
    )
    return gt

def evaluate_rankings(pred_df, gt_df, k_list=(5, 10)):
    rows = []
    merged = pred_df.merge(gt_df, on="dealer_id", how="inner")

    for _, r in merged.iterrows():
        dealer_id = r["dealer_id"]
        pred_items = r["pred_items"]
        true_items = r["true_items"]
        for k in k_list:
            rows.append({
                "dealer_id": dealer_id,
                "K": k,
                "recall": recall_at_k(pred_items, true_items, k),
                "ndcg": ndcg_at_k(pred_items, true_items, k),
            })

    score_df = pd.DataFrame(rows)
    summary = score_df.groupby("K")[["recall", "ndcg"]].mean().reset_index()
    return score_df, summary


# cutoff experiment

In [81]:
# Use the same cutoffs logic from before
min_time = views_dedup["created_time"].min()
max_time = views_dedup["created_time"].max()
valid_max_cutoff = max_time - timedelta(days=HORIZON_DAYS)

# 1 cutoffs to have enough data
cutoffs = pd.date_range(
    start=valid_max_cutoff, 
    end=valid_max_cutoff, 
    periods=1
)

cutoffs

DatetimeIndex(['2026-08-10 15:25:00+00:00'], dtype='datetime64[ns, UTC]', freq=None)

In [82]:
cutoff = cutoffs[0]

history_views, future_views = split_by_cutoff(views_dedup, cutoff, HORIZON_DAYS)
history_bids, future_bids = split_by_cutoff(bids_dedup, cutoff, HORIZON_DAYS)

future_labels = build_future_labels(future_views)
gt_df = build_ground_truth(future_views)

print("history_views:", history_views.shape)
print("future_views :", future_views.shape)
print("history_bids :", history_bids.shape)
print("future_bids  :", future_bids.shape)
print("future_labels:", future_labels.shape)
print("gt dealers   :", gt_df.shape[0])


history_views: (2714832, 8)
future_views : (414246, 8)
history_bids : (885434, 8)
future_bids  : (109247, 8)
future_labels: (105672, 4)
gt dealers   : 7444


# Build candidate set E2

In [83]:
candidates_e2 = build_e2_candidates(history_views)
print("E2 candidates:", candidates_e2.shape)
display(candidates_e2.head())

E2 candidates: (623990, 3)


,dealer_id,brand_id,model_id
0,3,187,1155
1,3,100,599
2,3,136,683
3,3,140,1065
4,3,100,595


# Score A2 on candidates

In [84]:
a2_scores = score_a2_popularity(history_views)

cand_a2 = candidates_e2.merge(a2_scores, on=ITEM_KEY_COLS, how="left")
cand_a2["score_a2"] = cand_a2["score_a2"].fillna(0)

# attach item_key for evaluation
cand_a2["item_key"] = list(zip(cand_a2["brand_id"], cand_a2["model_id"]))

pred_a2 = (
    cand_a2.sort_values(["dealer_id", "score_a2"], ascending=[True, False])
    .groupby("dealer_id")
    .agg(pred_items=("item_key", list))
    .reset_index()
)

score_rows_a2, summary_a2 = evaluate_rankings(pred_a2, gt_df, K_LIST)
display(summary_a2)


,K,recall,ndcg
0,5,0.211061,0.536000
1,10,0.321246,0.504322


# Score A1 on candidates

In [85]:
a1_scores = score_a1_dealer_history(history_views, history_bids, cutoff)

cand_a1 = candidates_e2.merge(a1_scores, on=["dealer_id"] + ITEM_KEY_COLS, how="left")
cand_a1["score_a1"] = cand_a1["score_a1"].fillna(cand_a1["score_a1"].min() if cand_a1["score_a1"].notna().any() else 0)
cand_a1["item_key"] = list(zip(cand_a1["brand_id"], cand_a1["model_id"]))

pred_a1 = (
    cand_a1.sort_values(["dealer_id", "score_a1"], ascending=[True, False])
    .groupby("dealer_id")
    .agg(pred_items=("item_key", list))
    .reset_index()
)

score_rows_a1, summary_a1 = evaluate_rankings(pred_a1, gt_df, K_LIST)
display(summary_a1)


,K,recall,ndcg
0,5,0.226656,0.569433
1,10,0.356049,0.555725


# Score A3 on candidates

In [86]:
cand_a3 = cand_a1.merge(
    a2_scores,
    on=ITEM_KEY_COLS,
    how="left"
)

cand_a3["score_a2"] = cand_a3["score_a2"].fillna(0)
cand_a3 = score_a3_blend(cand_a3, alpha=0.7)
cand_a3["item_key"] = list(zip(cand_a3["brand_id"], cand_a3["model_id"]))

pred_a3 = (
    cand_a3.sort_values(["dealer_id", "score_a3"], ascending=[True, False])
    .groupby("dealer_id")
    .agg(pred_items=("item_key", list))
    .reset_index()
)

score_rows_a3, summary_a3 = evaluate_rankings(pred_a3, gt_df, K_LIST)
display(summary_a3)


,K,recall,ndcg
0,5,0.212997,0.542547
1,10,0.324882,0.511793


# Compare results

In [87]:
summary_a1["model"] = "A1"
summary_a2["model"] = "A2"
summary_a3["model"] = "A3"

final_summary = pd.concat([summary_a1, summary_a2, summary_a3], ignore_index=True)
final_summary = final_summary[["model", "K", "recall", "ndcg"]].sort_values(["K", "model"])
display(final_summary)

,model,K,recall,ndcg
0,A1,5,0.226656,0.569433
2,A2,5,0.211061,0.536000
4,A3,5,0.212997,0.542547
1,A1,10,0.356049,0.555725
3,A2,10,0.321246,0.504322
5,A3,10,0.324882,0.511793
